In [ ]:
import numpy as np
import pandas as pd

## $\chi^2$ test for independence

In [ ]:
# iris dataset
path = "https://raw.githubusercontent.com/Armagaan/noc26_cs86/refs/heads/main/data/iris.csv"
df = pd.read_csv(path)
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [ ]:
# let's compare the length/width against their median values
df = df[["sepal_length", "sepal_width"]]
df.head()

,sepal_length,sepal_width
0,5.1,3.5
1,4.9,3.0
2,4.7,3.2
3,4.6,3.1
4,5.0,3.6


In [ ]:
length_median = df.sepal_length.median()
width_median = df.sepal_width.median()
print(length_median, width_median)

5.8 3.0


In [ ]:
df["sepal_length"] = (df["sepal_length"] > length_median).astype(int)
df["sepal_width"] = (df["sepal_width"] > width_median).astype(int)

df.head()

,sepal_length,sepal_width
0,0,1
1,0,0
2,0,1
3,0,1
4,0,1


In [ ]:
# short, short
# short, long
# long, short
# long, long

cross_tab = pd.pivot_table(
    data=df,
    index="sepal_length",
    columns="sepal_width",
    aggfunc=len
).values

print(cross_tab)

[[38 42]
 [45 25]]


In [ ]:
from scipy.stats import chi2_contingency

chi, pval, dof, exp_cross_tab = chi2_contingency(cross_tab, correction=False)

In [ ]:
alpha = 0.05
print(pval)

if pval < alpha:
    print("Reject the null hypothesis of columns being independent")
else:
    print("Fail to reject the null hypothesis")

0.03911091585039266
Reject the null hypothesis of columns being independent


## $\chi^2$ goodness of fit against normal distribution

In [ ]:
path = "https://raw.githubusercontent.com/Armagaan/noc26_cs86/refs/heads/main/data/temperature_distribution.csv"
df = pd.read_csv(path)
df.head()

,temp
0,31.052086
1,21.614336
2,6.885481
3,30.519586
4,23.033692


In [ ]:
H0 = "Population is normal"
HA = "Population isn't normal"

alhpa = 0.05

In [ ]:
len(df)

365

In [ ]:
# create bins based on the normal distribution so that a data point has equal probability of falling into any bin

from scipy.stats import norm

mean = df.temp.mean()
std = df.temp.std()

num_intervals = 12
# note: total area under curve is 1.
area_under_first_bin = 1 / num_intervals

bins = []
for i in range(1, num_intervals + 1):
    val = norm.ppf(i * area_under_first_bin, loc=mean, scale=std)
    bins.append(val)

bins

[np.float64(6.884466481964834),
 np.float64(10.842516800525793),
 np.float64(13.632496159304587),
 np.float64(15.954170247028785),
 np.float64(18.05236977494287),
 np.float64(20.05655929815203),
 np.float64(22.06074882136119),
 np.float64(24.158948349275278),
 np.float64(26.480622436999475),
 np.float64(29.27060179577827),
 np.float64(33.228652114339226),
 np.float64(inf)]

In [ ]:
def bin_element(x, bins):
    # assumed bins are sorted.
    for i, bin in enumerate(bins):
        if x <= bin:
            return i

In [ ]:
df["bins"] = df["temp"].apply(lambda x: bin_element(x, bins))
df.head()

,temp,bins
0,31.052086,10
1,21.614336,6
2,6.885481,1
3,30.519586,10
4,23.033692,7


In [ ]:
# cross table
# how many data instances fall under each bin
# df.groupby("bins").count().values.flatten()
cross_tab = df.groupby("bins").count().values.flatten()

# compare that against the expected number of data instances in each bin
exp_cross_tab = np.ones_like(cross_tab) * (len(df) // num_intervals)

In [ ]:
print(cross_tab)
print(exp_cross_tab)

[27 35 37 31 32 27 30 28 25 33 32 28]
[30 30 30 30 30 30 30 30 30 30 30 30]


In [ ]:
from scipy.stats import chisquare

chi, pval = chisquare(f_obs=cross_tab, f_exp=exp_cross_tab, ddof=2, sum_check=False)

print(pval)

if pval < alpha:
    print("Reject H0:", H0)
else:
    print("Fail to reject H0:", H0)

0.8541556856458349
Fail to reject H0: Population is normal
